In [ ]:
# ==============================================================
#  INPUTS — completar con los valores de tu robot
# ==============================================================

# --- Input 1: Conexión ---
PORT = '/dev/ttyUSB0'   # cambiar si es /dev/ttyTHS1
BAUD = 1000000

# --- Input 2-6: Poses del ciclo [q1, q2, q3, q4, q5, q6] en grados ---
# Registradas desde el robot real con mc.get_angles()
# Usa la Celda 2 (captura) para obtener estos valores
POSES = {
    'init_pose':   [0, 0, 0, 0, 0, 0],                           # Input 2
    'pick_upper':  [121.2, -9.58, -66.53, -11.07, -3.86, 70.75], # Input 3
    'pick_grasp':  [120.32,-31.72, -92.02,  34.27, -3.07, 69.78],# Input 4
    'place_upper': [-83.67,-14.41, -72.33,  -1.4,  -2.46, 44.20],# Input 5
    'place_grasp': [-82.61,-24.87, -98.52,  38.23, -3.16, 44.29],# Input 6
}

# --- Input 7: Parámetros de operación ---
MOVE_SPEED    = 90    # velocidad de movimiento (0-100)
GRIPPER_SPEED = 50    # velocidad del gripper
WAIT_MOVE     = 1.5   # segundos de espera tras send_angles
WAIT_GRIP     = 1.0   # segundos de espera tras set_gripper_state
MAX_RETRIES   = 3     # intentos ante fallo de comunicación
N_CICLOS      = 5     # cantidad de ciclos consecutivos

# ==============================================================
print('INPUTS cargados:')
print(f'  Puerto      : {PORT}')
print(f'  Velocidad   : {MOVE_SPEED}')
print(f'  Ciclos      : {N_CICLOS}')
print()
for nombre, angulos in POSES.items():
    print(f'  {nombre:<14}: {angulos}')

---
## Celda 2 — Conexión y captura de poses

Mueve el brazo a la posición que quieras registrar, cambia `POSE_A_CAPTURAR` y ejecuta.

In [ ]:
import time
import numpy as np
from datetime import datetime
from pymycobot.mycobot import MyCobot

mc = MyCobot(PORT, BAUD)
mc.power_on()
time.sleep(1)
print(f'Conectado en {PORT}')

# --- Captura de una pose: mover el brazo y ejecutar ---
POSE_A_CAPTURAR = 'pick_upper'  # <-- cambiar al nombre que necesitas

time.sleep(0.3)
angulos = mc.get_angles()
coords  = mc.get_coords()
time.sleep(0.2)

POSES[POSE_A_CAPTURAR] = list(angulos)

print(f'\nPose capturada -> "{POSE_A_CAPTURAR}"')
print(f'  Ángulos : {[round(a, 2) for a in angulos]}')
if coords:
    print(f'  Coords  : x={coords[0]:.1f}  y={coords[1]:.1f}  z={coords[2]:.1f} mm')
print(f'\nPOSES actualizado:')
for k, v in POSES.items():
    print(f'  {k:<14}: {v}')

---
## Celda 3 — Lógica del ciclo (no modificar)

### Pregunta 2: ciclo de agarre con `send_angles` + `set_gripper_state`
### Pregunta 5: gestión de errores con reintentos

In [ ]:
# --------------------------------------------------------------
# Pregunta 5: reintentos ante fallos de comunicación
# --------------------------------------------------------------
def send_with_retry(func, *args):
    for intento in range(1, MAX_RETRIES + 1):
        try:
            return func(*args)
        except Exception as e:
            print(f'    [REINTENTO {intento}/{MAX_RETRIES}] {func.__name__}: {e}')
            time.sleep(1.0)
    raise RuntimeError(f'Fallo tras {MAX_RETRIES} intentos: {func.__name__}')


# --------------------------------------------------------------
# Pregunta 2: secuencia de poses
# Cada elemento es una pose [q1..q6] o 'open' / 'close'
# --------------------------------------------------------------
SECUENCIA = [
    POSES['init_pose'],    # 1. reposo
    POSES['pick_upper'],   # 2. sobre objeto A
    POSES['pick_grasp'],   # 3. bajar a A
    'close',               # 4. cerrar gripper
    POSES['pick_upper'],   # 5. subir con objeto
    POSES['init_pose'],    # 6. clearance intermedio
    POSES['place_upper'],  # 7. sobre zona B
    POSES['place_grasp'],  # 8. bajar a B
    'open',                # 9. abrir gripper
    POSES['place_upper'],  # 10. subir sin objeto
    POSES['init_pose'],    # 11. volver a reposo
]


def ejecutar_paso(paso):
    """Ejecuta un paso de la secuencia: ángulos o comando de gripper."""
    if paso == 'close':
        send_with_retry(mc.set_gripper_state, 1, GRIPPER_SPEED)
        time.sleep(WAIT_GRIP)
    elif paso == 'open':
        send_with_retry(mc.set_gripper_state, 0, GRIPPER_SPEED)
        time.sleep(WAIT_GRIP)
    else:
        send_with_retry(mc.send_angles, paso, MOVE_SPEED)
        time.sleep(WAIT_MOVE)


def run_ciclo(num):
    """
    Ejecuta un ciclo completo A->B.
    Retorna dict con resultado por paso.
    """
    t0 = time.time()
    paso_fallido = None

    for i, paso in enumerate(SECUENCIA):
        nombre = paso if isinstance(paso, str) else f'pose_{i+1}'
        print(f'    Paso {i+1:>2}/{len(SECUENCIA)}: {nombre}')
        try:
            ejecutar_paso(paso)
        except RuntimeError as e:
            print(f'    ERROR en paso {i+1}: {e}')
            paso_fallido = i + 1
            # recuperar a reposo
            try:
                mc.send_angles(POSES['init_pose'], 50)
            except Exception:
                pass
            break

    exito = paso_fallido is None
    return {
        'ciclo':        num,
        'exito':        exito,
        'paso_fallido': paso_fallido,
        'tiempo':       round(time.time() - t0, 2),
    }


print('Lógica del ciclo lista.')
print(f'Pasos por ciclo: {len(SECUENCIA)}')

---
## Celda 4 — Calibración de velocidad y tiempos

### Pregunta 3: calibrar velocidades y tiempos de espera

In [ ]:
# Prueba el movimiento init -> pick_upper a distintas velocidades
# y mide cuánto tarda realmente usando is_moving()

velocidades_prueba = [50, 70, 90]

print('=' * 52)
print('CALIBRACIÓN — init_pose → pick_upper')
print(f'{"Velocidad":>10}  {"T real (s)":>12}  {"Estado":>8}')
print('-' * 52)

resultados_cal = []
for vel in velocidades_prueba:
    mc.send_angles(POSES['init_pose'], 50)
    time.sleep(2.5)

    t0 = time.time()
    estado = 'OK'
    try:
        mc.send_angles(POSES['pick_upper'], vel)
        time.sleep(0.4)
        for _ in range(40):          # espera hasta que deje de moverse
            if mc.is_moving() == 0:
                break
            time.sleep(0.2)
    except Exception as e:
        estado = f'ERR'

    t_real = round(time.time() - t0, 2)
    resultados_cal.append((vel, t_real, estado))
    print(f'{vel:>10}  {t_real:>12.2f}  {estado:>8}')

print('=' * 52)

# Usar la velocidad más alta sin error
ok_vels = [v for v, t, e in resultados_cal if e == 'OK']
if ok_vels:
    MOVE_SPEED = max(ok_vels)
    # Estimar tiempo de espera como el real + 10% de margen
    t_mayor = max(t for v, t, e in resultados_cal if e == 'OK' and v == MOVE_SPEED)
    WAIT_MOVE = round(t_mayor * 1.1, 1)

print(f'\nVelocidad seleccionada : {MOVE_SPEED}')
print(f'Tiempo de espera (WAIT_MOVE) ajustado a: {WAIT_MOVE}s')

mc.send_angles(POSES['init_pose'], 50)
time.sleep(2.5)

---
## Celda 5 — Prueba unitaria (1 ciclo)

Verifica que las poses y el gripper funcionan antes de los 5 ciclos.

In [ ]:
print('=== PRUEBA UNITARIA (1 ciclo) ===')
mc.send_angles(POSES['init_pose'], MOVE_SPEED)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

resultado_prueba = run_ciclo(0)

print()
print(f'Resultado : {"OK" if resultado_prueba["exito"] else "FALLO"}')
print(f'Tiempo    : {resultado_prueba["tiempo"]}s')
if not resultado_prueba['exito']:
    print(f'Falló en paso {resultado_prueba["paso_fallido"]} — revisar pose o gripper')

---
## Celda 6 — 5 ciclos consecutivos

### Pregunta 4: ejecutar 5 ciclos y registrar tasa de éxito

In [ ]:
log = []

mc.send_angles(POSES['init_pose'], MOVE_SPEED)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

print('=' * 55)
print(f'INICIO {N_CICLOS} CICLOS — {datetime.now().strftime("%H:%M:%S")}')
print('=' * 55)

for n in range(1, N_CICLOS + 1):
    print(f'\n--- Ciclo {n}/{N_CICLOS} ---')
    resultado = run_ciclo(n)
    log.append(resultado)
    estado = 'OK' if resultado['exito'] else f'FALLO (paso {resultado["paso_fallido"]})'
    print(f'  -> {estado}  ({resultado["tiempo"]}s)')
    if not resultado['exito']:
        time.sleep(3)              # pausa antes del siguiente ciclo

print('\n' + '=' * 55)
print('FIN DE CICLOS')
print('=' * 55)

---
## Celda 7 — Reporte de métricas

In [ ]:
n_ok   = sum(1 for r in log if r['exito'])
tasa   = n_ok / len(log) * 100 if log else 0
t_prom = sum(r['tiempo'] for r in log) / len(log) if log else 0

print('=' * 55)
print('REPORTE FINAL — P5 Control de Trayectorias')
print('=' * 55)
print(f'{"Ciclo":<8} {"Resultado":<14} {"Tiempo (s)":<12} {"Paso fallido"}')
print('-' * 55)
for r in log:
    res  = 'OK' if r['exito'] else 'FALLO'
    paso = '-' if r['exito'] else str(r['paso_fallido'])
    print(f'  {r["ciclo"]:<6} {res:<14} {r["tiempo"]:<12} {paso}')
print('=' * 55)
print(f'Exitosos      : {n_ok}/{len(log)}  ({tasa:.0f}%)')
print(f'Tiempo prom.  : {t_prom:.1f}s/ciclo')
print()
veredicto = 'APROBADO' if tasa >= 80 else 'REQUIERE AJUSTE'
print(f'Resultado: {veredicto}  (criterio >= 80%)')
print('=' * 55)